In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import to_date, trim, col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import round, col
from pyspark.sql.types import DecimalType



In [0]:
df_bronze = spark.table("liquid_telecom.bronze.b_fact_subscriptions_revenue")
display(df_bronze.limit(10))

product_id,account_id,currency,total_one_off_price,total_recurring_price,total_contract_value,operating_country,account_created_date,order_date,_source_file,ingested_at
P000000,A000000000000,RWF,0,0,0,LTR - Liquid Telecommunications Rwanda Ltd,31-07-2024,02-08-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P000000,A000000000000,RWF,0,0,0,LTR - Liquid Telecommunications Rwanda Ltd,31-07-2024,02-08-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P002223,A000000000001,USD,720,45,0,null,24-08-2024,24-08-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P002223,A000000000001,USD,720,45,1260,null,24-08-2024,24-08-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P005981,A000000000001,USD,180,168,0,null,28-11-2024,28-11-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P005981,A000000000001,USD,180,168,2196,null,28-11-2024,28-11-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P002224,A000000000001,USD,720,35,0,null,24-08-2024,24-08-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P002224,A000000000001,USD,720,35,1140,null,24-08-2024,24-08-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P002225,A000000000001,USD,720,19,0,null,21-07-2024,21-07-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z
P002225,A000000000001,USD,720,19,948,null,21-07-2024,21-07-2024,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z


### **Changing date string type to date type**

In [0]:
df = df_bronze \
    .withColumn("account_created_date", to_date(trim(col("account_created_date")), "dd-MM-yyyy")) \
    .withColumn("order_date", to_date(trim(col("order_date")), "dd-MM-yyyy"))

### **Dropping duplicate rows where all the other column values are same and total contract value is zero**

In [0]:
window_spec = Window.partitionBy(
    "product_id", "account_id", "order_date"
).orderBy(col("total_contract_value").desc())

df = df.withColumn("rn", row_number().over(window_spec)) \
       .filter(col("rn") == 1) \
       .drop("rn")


### **Converting all the currencies in to USD**

In [0]:
df.select("currency").distinct().show()


+--------+
|currency|
+--------+
|     RWF|
|     USD|
|     ZMW|
|     ZWL|
|     KES|
|     EUR|
|     UGX|
|     ZAR|
|     TZS|
|     NGN|
|     BWP|
+--------+



In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "rate_to_usd",
    when(col("currency") == "USD", 1.0)
    .when(col("currency") == "EUR", 1.08)
    .when(col("currency") == "ZAR", 0.054)
    .when(col("currency") == "KES", 0.0073)
    .when(col("currency") == "NGN", 0.00067)
    .when(col("currency") == "UGX", 0.00027)
    .when(col("currency") == "TZS", 0.00040)
    .when(col("currency") == "RWF", 0.00079)
    .when(col("currency") == "ZMW", 0.037)
    .when(col("currency") == "ZWL", 0.0028)
    .when(col("currency") == "BWP", 0.074)
)

df = df.withColumn(
        "total_one_off_price_usd",
        col("total_one_off_price") * col("rate_to_usd")
    ).withColumn(
        "total_recurring_price_usd",
        col("total_recurring_price") * col("rate_to_usd")
    ).withColumn(
        "total_contract_value_usd",
        col("total_contract_value") * col("rate_to_usd")
    )

df = df.withColumn(
        "total_one_off_price_usd",
        round(col("total_one_off_price_usd"), 2)
    ).withColumn(
        "total_recurring_price_usd",
        round(col("total_recurring_price_usd"), 2)
    ).withColumn(
        "total_contract_value_usd",
        round(col("total_contract_value_usd"), 2)
    )

In [0]:
df = df.withColumn(
    "total_contract_value_usd",
    col("total_contract_value_usd").cast(DecimalType(18,2))
)
df = df.withColumn(
    "total_one_off_price_usd",
    col("total_one_off_price_usd").cast(DecimalType(18,2))
)
df = df.withColumn(
    "total_recurring_price_usd",
    col("total_recurring_price_usd").cast(DecimalType(18,2))
)
df = df.withColumn(
    "total_contract_value",
    col("total_contract_value").cast(DecimalType(18,2))
)
df = df.withColumn(
    "total_one_off_price",
    col("total_one_off_price").cast(DecimalType(18,2))
)
df = df.withColumn(
    "total_recurring_price",
    col("total_recurring_price").cast(DecimalType(18,2))
)

In [0]:
df = df.filter(
    ~(
        (col("total_one_off_price") == 0) &
        (col("total_recurring_price") == 0) &
        (col("total_contract_value") == 0)
    )
)

In [0]:
df.filter(col("operating_country").isNull()).show(truncate=False)

+----------+-------------+--------+-------------------+---------------------+--------------------+-----------------+--------------------+----------+--------------------------------------------------------------------------------+--------------------------+-----------+-----------------------+-------------------------+------------------------+
|product_id|account_id   |currency|total_one_off_price|total_recurring_price|total_contract_value|operating_country|account_created_date|order_date|_source_file                                                                    |ingested_at               |rate_to_usd|total_one_off_price_usd|total_recurring_price_usd|total_contract_value_usd|
+----------+-------------+--------+-------------------+---------------------+--------------------+-----------------+--------------------+----------+--------------------------------------------------------------------------------+--------------------------+-----------+-----------------------+--------------------

### **where operating country is null and empty treating that as not available**

In [0]:
df = df.withColumn(
    "operating_country",
    when(
        col("operating_country").isNull() | (col("operating_country") == ""),
        "Not Available"
    ).otherwise(col("operating_country"))
)

In [0]:
display(df.limit(10))

product_id,account_id,currency,total_one_off_price,total_recurring_price,total_contract_value,operating_country,account_created_date,order_date,_source_file,ingested_at,rate_to_usd,total_one_off_price_usd,total_recurring_price_usd,total_contract_value_usd
P000001,A000000000008,USD,0.00,100.00,1200.00,LTSAT - Liquid Telecommunications Satellite Services,2024-05-12,2024-05-12,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,1.0,0.00,100.00,1200.00
P000002,A000000000022,ZMW,1365.00,680.00,9525.00,LTZM - Liquid Telecommunications Zambia Ltd,2024-10-13,2024-10-16,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,0.037,50.50,25.16,352.42
P000002,A000000000477,ZMW,1172.45,0.00,1172.45,LTZM - Liquid Telecommunications Zambia Ltd,2024-09-10,2024-10-03,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,0.037,43.38,0.00,43.38
P000003,A000000000008,USD,0.00,275.00,3300.00,LTSAT - Liquid Telecommunications Satellite Services,2024-06-09,2024-06-12,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,1.0,0.00,275.00,3300.00
P000004,A000000000008,USD,0.00,275.00,3300.00,LTSAT - Liquid Telecommunications Satellite Services,2024-05-12,2024-05-12,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,1.0,0.00,275.00,3300.00
P000005,A000000000047,USD,270.00,0.00,270.00,LTK - Liquid Telecommunications Kenya Ltd.,2024-10-27,2024-10-30,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,1.0,270.00,0.00,270.00
P000006,A000000000048,ZWL,4800000.00,3750000.00,49800000.00,LTZ - Liquid Telecom Zimbabwe,2024-12-29,2025-01-02,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,0.0028,13440.00,10500.00,139440.00
P000007,A000000000008,USD,0.00,40.00,480.00,LTSAT - Liquid Telecommunications Satellite Services,2024-04-16,2024-04-17,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,1.0,0.00,40.00,480.00
P000007,A000000000008,USD,0.00,80.00,960.00,LTSAT - Liquid Telecommunications Satellite Services,2024-06-09,2024-06-12,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,1.0,0.00,80.00,960.00
P000009,A000000000063,KES,10000.00,7000.00,94000.00,LTK - Liquid Telecommunications Kenya Ltd.,2024-01-23,2024-05-12,dbfs:/Volumes/liquid_telecom/raw/liquid_telecom_volume/fact/fact_orders_2024.csv,2026-01-30T07:37:47.381Z,0.0073,73.00,51.10,686.20


### **saving to silver schema in delta format**

In [0]:
catalog = "liquid_telecom"
database = "silver"
table = "s_fact_subscriptions_revenue"

full_table_name = f"{catalog}.{database}.{table}"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {database}")

if not spark.catalog.tableExists(full_table_name):
    print("Creating Silver fact table")

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_table_name)

else:
    print("Merging into existing Silver fact table")

    deltaTable = DeltaTable.forName(spark, full_table_name)

    deltaTable.alias("silver") \
        .merge(
            df.alias("batch"),
            """
            silver.product_id = batch.product_id
            AND silver.account_id = batch.account_id
            AND silver.order_date = batch.order_date
            """
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()


Merging into existing Silver fact table
